In [23]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest, proportion_effectsize
from statsmodels.stats.power import NormalIndPower

np.random.seed(42)

Task 1: Chi-Square Goodness-of-Fit Test

H0 : players choose equally from all character classes
H1 : choices are not equal-at least one of them differs

In [24]:
#Observed counts 
observed = np.array([120, 85, 110, 85])

#Expected counts
total_players = observed.sum()
expected = np.array([total_players*0.25]*4)

observed, expected

(array([120,  85, 110,  85]), array([100., 100., 100., 100.]))

In [25]:
#Apply chi-square test 
chi2_stat, p_value = st.chisquare(observed, f_exp=expected)
chi2_stat, p_value

(np.float64(9.5), np.float64(0.023331360430831553))

In [26]:
degrees_of_freedom= len(observed)-1
degrees_of_freedom

3

In [27]:
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {degrees_of_freedom}")
print(f"P-value: {p_value:.4f}")

Chi-Square Statistic: 9.5000
Degrees of Freedom: 3
P-value: 0.0233


In [28]:
#Decision 
alpha = 0.05
if p_value < alpha:
    print('Rejecting H0 - there is significant difference between choices')
else:
    print("Fail to reject H0 - DIfferences can be coincidence")  

Rejecting H0 - there is significant difference between choices


We expected 100 for each class but, 
Warrior - 120 We expected 100 but got 100 instead, the difference is 120-100 = 20 so it is 20% over-represented 
Rogue - it was chosen by 110 player which is 10% higher than the expected - 10% over-represented
Mage and Healer: Both states 85 player which is lower than the expexted which is 100. 85 - 100 = -15 So 15% under-represented 

Task 2: Chi-Square Test of Independence 

Seeing if there is any relation between two categories. We need to determine whether subscription tier is associated with churn status 

H0 : there is no relation between these two value. 
H1 : There is a statistical relation between the values 

In [29]:
#Contingency table 
#Column -> Subscription tier (free, basic, premium)
#Row ->  Churn status (churned or retained)

In [30]:
data = np.array([
    [90, 60, 30],
    [110, 140, 170]])
data

array([[ 90,  60,  30],
       [110, 140, 170]])

In [31]:
#create contingency table
columns = ['Free', 'Basic', 'Premium']
index = ['Churned', 'Retained']
table = pd.DataFrame(data, index=index, columns=columns)
table

,Free,Basic,Premium
Churned,90,60,30
Retained,110,140,170


In [32]:
#Run chi2 
chi2_stat, p_value, dof, expected= st.chi2_contingency(table)
expected

array([[ 60.,  60.,  60.],
       [140., 140., 140.]])

In [33]:
expected_df = pd.DataFrame(expected, index=index, columns=columns)

print('-------Expected Frequencies Table -----')
print(expected_df)
print("\n--- Statistics ---")
print(f"Chi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"P-value: {p_value:.4e}")

-------Expected Frequencies Table -----
           Free  Basic  Premium
Churned    60.0   60.0     60.0
Retained  140.0  140.0    140.0

--- Statistics ---
Chi-Square Statistic: 42.8571
Degrees of Freedom: 2
P-value: 4.9396e-10


In [34]:
p_value

np.float64(4.939576018831198e-10)

In [35]:
alpha = 0.05
if p_value < alpha:
    decision = "Reject H0"
    interpretation = "There is a strong relation between subscription type and customer churn"
else:
    decision = "Fail to reject H0"
    interpretation = "There is no relation between subscription type and customer churn"

print(f"\nDecision: {decision}")
print(f"Interpretation: {interpretation}")


Decision: Reject H0
Interpretation: There is a strong relation between subscription type and customer churn


There is a very strong correlation between subscription type and customer churn. The numbers show that as the subscription level increases (Free -> Basic -> Premium), customer churn decreases and retention increases. So, the marketing team can say that switching to a Premium package increases the likelihood of customer retention.

Task 3: Compute Cramer's V


The Chi-Square test only tells us whether a relationship exists (its statistical significance). It does not tell us how strong that relationship is. Cramér's V fills this gap:

It takes a value between 0 and 1.

Values ​​close to 0 indicate a weak relationship, and values ​​close to 1 indicate a very strong relationship.

In [36]:
def cramers_v(chi2, n, min_dim):
    """
    Cramér's V hesablayan funksiya.
    chi2: Chi-square statistikası
    n: Ümumi müşahidə sayı (total sample size)
    min_dim: min(sətir, sütun) - 1
    """
    return np.sqrt(chi2 / (n * min_dim))

In [37]:
#entering values from task 2
chi2_task2 = chi2_stat
n_total = 600


#we had a 2x3 table 
#min(2, 3) = 2 min_dim = 2-1=1
min_dim_val = 1


In [38]:
v_stat = cramers_v(chi2_task2, n_total, min_dim_val)

print(f"Cramér's V: {v_stat:.4f}")

Cramér's V: 0.2673


In [39]:
if v_stat < 0.20:
    effect = "Small (Zəif)"
elif v_stat < 0.40:
    effect = "Medium (Orta)"
else:
    effect = "Large (Güclü)"

print(f"Effect Size (Cohen's Benchmark): {effect}")

Effect Size (Cohen's Benchmark): Medium (Orta)


The Connection Is Real: The relationship between subscription plan and customer churn is not just a coincidence; it is a statistically proven trend.
Impact: This "medium" level of connection means that upselling a customer from a Free plan to a Premium plan significantly changes their likelihood of staying in the system.
Strategy: The marketing team should focus on campaigns to convert Free users to Basic or Premium, as this conversion is one of the main factors that directly reduces the risk of "Churn". However, since the connection is not above 0.50 (very strong), they should also understand that "package type" is not the only reason for churn (for example, customer service or application errors can also play a role).
